In [10]:
import pandas as pd

uploaded_file = "C:/Users/mzigm/Downloads/donations.csv"
df = pd.read_csv(uploaded_file, dtype={"zip": str , "Zip/Postal Code": str, "phone": str, "Contact Phone Number": str}, keep_default_na=False)
df['date_time'] = pd.to_datetime(df['date_time'], format='mixed')
df.to_csv("C:/Users/mzigm/Downloads/new_donations.csv" , index=False)

In [1]:
import sqlite3
import os
import re

class _StaticCursor:
    def __init__(self, rows):
        self._rows = rows

    def fetchall(self):
        return self._rows


class FilteringConnection(sqlite3.Connection):
    def execute(self, sql, parameters=()):
        if sql.strip().startswith("SELECT name, sql FROM sqlite_master"):
            rows = super().execute(sql, parameters).fetchall()
            filtered = []
            for name, create_sql in rows:
                if "temp_" in name or "backup_" in name:
                    continue
                if name == "trans_source":
                    data_sql = self._dump_table_data(name)
                    if data_sql:
                        create_sql = create_sql.strip() + ";\n" + data_sql
                create_sql = self._ensure_not_null(create_sql, name)
                filtered.append((name, create_sql))
            return _StaticCursor(filtered)
        return super().execute(sql, parameters)

    def _ensure_not_null(self, create_sql, table_name):
        info = super().execute(f'PRAGMA table_info("{table_name}")').fetchall()
        not_null_cols = {row[1] for row in info if row[3] == 1}
        if not not_null_cols:
            return create_sql
        open_paren = create_sql.find("(")
        close_paren = create_sql.rfind(")")
        if open_paren == -1 or close_paren == -1 or close_paren <= open_paren:
            return create_sql
        prefix = create_sql[: open_paren + 1]
        body = create_sql[open_paren + 1 : close_paren]
        suffix = create_sql[close_paren:]
        parts = self._split_columns(body)
        updated_parts = []
        for part in parts:
            part_strip = part.strip()
            lower_part = part_strip.lower()
            if lower_part.startswith(("constraint ", "foreign key", "primary key", "unique", "check")):
                updated_parts.append(part_strip)
                continue
            match = re.match(r'^\s*("[^"]+"|\S+)', part_strip)
            if not match:
                updated_parts.append(part_strip)
                continue
            col_name = match.group(1).strip('"')
            if col_name in not_null_cols and "not null" not in lower_part:
                part_strip = f"{part_strip} NOT NULL"
            updated_parts.append(part_strip)
        joined = ",\n    ".join(updated_parts)
        return f"{prefix}\n    {joined}\n{suffix}"

    @staticmethod
    def _split_columns(body):
        parts = []
        current = []
        depth = 0
        for char in body:
            if char == "(":
                depth += 1
            elif char == ")":
                depth -= 1
            if char == "," and depth == 0:
                parts.append("".join(current))
                current = []
            else:
                current.append(char)
        if current:
            parts.append("".join(current))
        return parts

    def _dump_table_data(self, table_name):
        cur = super().execute(f'SELECT * FROM "{table_name}"')
        rows = cur.fetchall()
        if not rows:
            return ""
        cols = ", ".join(f'"{d[0]}"' for d in cur.description)
        inserts = []
        for row in rows:
            values = ", ".join(self._quote_value(v) for v in row)
            inserts.append(f'INSERT INTO "{table_name}" ({cols}) VALUES ({values});')
        return "\n".join(inserts)

    @staticmethod
    def _quote_value(value):
        if value is None:
            return "NULL"
        if isinstance(value, (int, float)):
            return str(value)
        escaped_string = str(value).replace("'", "''")
        return f"'{escaped_string}'"


_original_connect = sqlite3.connect


def _filtered_connect(*args, **kwargs):
    kwargs["factory"] = FilteringConnection
    return _original_connect(*args, **kwargs)


sqlite3.connect = _filtered_connect

db_path = "/home/ajshen/Projects/sn_streamlit/sigma_nu_donations.db"
output_dir = "/home/ajshen/Projects/sn_streamlit/schema_sql"
os.makedirs(output_dir, exist_ok=True)

with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys=ON;")
    cursor = conn.execute(
        "SELECT name, sql FROM sqlite_master "
        "WHERE type='table' AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL;"
    )
    for table_name, create_sql in cursor.fetchall():
        file_path = os.path.join(output_dir, f"{table_name}.sql")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(create_sql.strip() + ";\n")